# 📝 과제 02: 온라인 쇼핑 상담 데이터셋 다듬기

## 만들 결과

과제 01과 같은 방식으로 만든 온라인 쇼핑 상담 Q&A에 기존 FAQ를 합친 고정 파일을, 교안 02의 순서대로 학습·검증 파일로 다듬습니다. 생성용 LLM의 결과는 실행마다 달라지므로 이 과제는 미리 만들어 둔 파일을 쓰고, 증강만 표본 10개로 직접 호출해 확인합니다.

<img src="images/41_gwaje02_pipeline.png" width="1050" alt="제목은 '과제 02: 온라인 쇼핑 상담 데이터셋 다듬기'입니다. 번호 배지 1~6의 카드 여섯 장이 왼쪽에서 오른쪽으로 화살표로 이어집니다. 1 '증강 · 숫자 확인'은 두 갈래로 나뉜 카드에 초록 체크입니다. 2 '품질 필터 네 단계'는 관문 네 개짜리 깔때기입니다. 3 '오염 검사'는 회색 카드와 겹친 묶음에 빨강 엑스입니다. 4 '구성 점검 · 균형'은 높이가 다른 막대 네 개와 점선이고, 가장 높은 막대가 점선에서 끝나며 그 위로 주황 카드 두 장이 빠져나가 빨강 엑스가 붙습니다. 5 'chat 포맷 · 정합'은 회색, 파랑, 초록 말풍선입니다. 6 '원본 단위 분할 · 저장'은 파랑과 노랑 묶음이 큰 상자에, 빨강 묶음이 작은 상자에 통째로 들어가고 각 상자 아래에 보라 파일이 있는 그림입니다.">

- 표본 질문 10개를 패러프레이즈와 역번역으로 늘리고, 원본과 숫자가 같은 행만 남깁니다.
- 품질 필터 네 단계로 198행을 183행으로 거르고, 단계마다 버린 행의 결함 종류를 읽습니다.
- 평가 질문 16개와 겹치는 원본 묶음을 빼 177행을 남깁니다.
- 방법, 주제, 같은 답 반복(묶음 크기)으로 구성을 점검하고, 주제 상한 48행을 넘는 주제를 줄여 169행으로 균형을 맞춥니다.
- 균형을 맞춘 행을 chat 포맷으로 바꿔 첫 행의 학습 텍스트가 추론 프롬프트로 시작하는지 확인하고, 학습 텍스트의 최대 토큰 수(122토큰)를 셉니다.
- 행 단위 분할의 누수(검증 34행 가운데 24행)를 확인한 뒤 원본 단위로 학습 135행, 검증 34행으로 나눕니다.
- 두 파일을 저장하고 범위 밖 비율과 주제 상한까지 적은 데이터 카드를 씁니다.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/commerce_qa_prepared.jsonl`(198행)
  - **파일 구성:** 전자상거래법 조문(국가법령정보센터 law.go.kr)으로 수업용으로 만든 Q&A입니다. 문서 생성 행과 범위 밖 거절 행 뒤에 패러프레이즈·역번역 행과 기존 FAQ 행이 합쳐져 있고, 행마다 원본 Q&A의 id인 `origin_id`가 있습니다.
  - **키:** `id`, `origin_id`, `doc_id`(기존 FAQ에는 없음), `source`, `method`, `question`, `answer`, `question_type`(문서 생성 행과 그 증강 행만)입니다.
  - **질문 늘리기 행을 넣지 않은 이유:** 과제 01의 질문 늘리기 행은 넣지 않았습니다. 근거 조문(`doc_id`)이 있는 원본 Q&A라 다듬는 단계에서 문서 생성 행과 똑같이 다뤄지므로, 원본 Q&A는 문서 생성 행 하나로 두었습니다.
  - **과제 01 결과와 다른 점:** 이 파일은 수업용으로 따로 만들어 번호를 붙인 고정 파일이라, `id` 모양(`c001`, `out-01`)과 범위 밖 행 수가 과제 01이 만든 `commerce_qa.jsonl`(`cqa-0001`, `out-0001`)과 다릅니다.
- **범위 밖 행:** 이 챗봇이 다루는 전자상거래법으로 답할 수 없는 질문에 고정 거절 답을 붙인 행입니다(`id` `out-01`부터, `doc_id` "없음", `source`와 `method` "범위 밖").
- **평가 질문:** `data/과제/commerce_eval_questions.json`(16개). 이 데이터셋으로 학습한 모델을 평가할 때 쓸 질문이라 학습·검증 어느 쪽에도 없어야 합니다.
- **제공 코드:** 준비 셀이 `commerce_rows`(고정 파일의 행 리스트), `commerce_eval_questions`, `SYSTEM_PROMPT_COMMERCE`, 토크나이저 `chat_tokenizer`, 교안 02의 함수(주제 상한 `cap_by_topic` 포함)와 난수 시드 `SPLIT_SEED`(42), 검증 비율 `VAL_RATIO`(0.2)를 만듭니다. `to_chat`의 기본 system 문장은 `SYSTEM_PROMPT_COMMERCE`입니다.
- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"  # 이 단원이 만든 데이터셋 파일을 저장하는 폴더
output_dir.mkdir(exist_ok=True)

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 데이터를 만드는 생성용 LLM. .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다
# use_responses_api=True: 앞 단원들과 같이 OpenAI Responses API로 호출합니다
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 테이블의 긴 글(답, 조문)이 50자에서 잘리지 않고 끝까지 보이게 합니다
pd.set_option("display.max_colwidth", None)

print("생성용 LLM:", model_name)

In [ ]:
# 교안 02의 JSONL 읽기와 쓰기, chat 포맷 함수입니다. system 문장은 온라인 쇼핑 상담 챗봇 문장입니다
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."


def load_jsonl(path):
    """JSONL 파일을 줄마다 읽어 행 딕셔너리(dict) 리스트로 돌려줍니다(빈 줄은 건너뜁니다)."""
    rows = []
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip() != "":
                rows.append(json.loads(line))
    return rows


def save_jsonl(rows, path):
    """행 딕셔너리 리스트를 한 줄에 JSON 하나씩 파일로 저장합니다."""
    with open(path, "w", encoding="utf-8") as file:
        for row in rows:
            # ensure_ascii=False 로 한글을 \uXXXX 대신 글자 그대로 씁니다
            file.write(f"{json.dumps(row, ensure_ascii=False)}\n")


def to_chat(row, system_prompt=SYSTEM_PROMPT_COMMERCE):
    """Q&A 한 행을 system, user, assistant 세 메시지의 chat 포맷 딕셔너리로 돌려줍니다."""
    return {"messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]}

In [ ]:
# 교안 02의 품질 필터 함수입니다
def format_ok(row):
    """question, answer 키가 모두 있고 두 값이 공백만 있는 문자열이 아니면 True를 돌려줍니다."""
    for key in ["question", "answer"]:
        if key not in row:
            return False
        if row[key].strip() == "":  # 공백만 있는 값도 빈 값으로 봄
            return False
    return True


def length_ok(row, min_a=5):
    """답의 글자 수가 min_a 이상이면 True를 돌려줍니다."""
    return len(row["answer"]) >= min_a


def dedup_exact(rows):
    """(질문, 답)이 처음 나온 행만 순서대로 남긴 리스트를 돌려줍니다."""
    seen = set()
    kept = []
    for row in rows:
        # (질문, 답)을 set에 기록해 두면 이미 본 쌍인지 바로 알 수 있습니다
        key = (row["question"], row["answer"])
        if key not in seen:
            seen.add(key)
            kept.append(row)
    return kept


def jaccard(a, b):
    """공백을 뺀 두 문장의 글자 집합으로 계산한 자카드 유사도를 돌려줍니다."""
    chars_a = set(a.replace(" ", ""))
    chars_b = set(b.replace(" ", ""))
    return len(chars_a & chars_b) / len(chars_a | chars_b)


def is_near_dup(a, b, threshold=0.8):
    """두 문장의 자카드 유사도가 threshold 이상이면 True를 돌려줍니다."""
    return jaccard(a, b) >= threshold


def dedup_similar(rows, threshold=0.8):
    """이미 남긴 질문과 유사 중복인 행을 버리고, 먼저 나온 행만 남긴 리스트를 돌려줍니다."""
    kept = []
    for row in rows:
        # 이미 남긴 질문 가운데 하나라도 비슷하면 이 행을 버립니다
        duplicated = False
        for kept_row in kept:
            if is_near_dup(row["question"], kept_row["question"], threshold):
                duplicated = True
                break
        if not duplicated:
            kept.append(row)
    return kept


def dropped_rows(before, after):
    """before 가운데 after에 없는 행(그 단계에서 버린 행)을 돌려줍니다."""
    # 남긴 id를 set으로 모읍니다
    kept_ids = {row["id"] for row in after}

    # before에서 그 id에 없는 행만 모읍니다
    dropped = []
    for row in before:
        if row["id"] not in kept_ids:
            dropped.append(row)
    return dropped

In [ ]:
# 교안 02의 오염 검사, 주제 상한, 분할 함수입니다
import random


def char_ngrams(text, n=3):
    """공백을 뺀 문장의 글자 n-gram 집합을 돌려줍니다."""
    compact = text.replace(" ", "")
    ngrams = set()
    for index in range(len(compact) - n + 1):  # n글자씩 한 칸씩 밀며 자름
        ngrams.add(compact[index:index + n])
    return ngrams


def ngram_overlap(question, eval_question):
    """평가 질문의 글자 3-gram 가운데 question에도 있는 비율을 돌려줍니다."""
    eval_ngrams = char_ngrams(eval_question)
    return len(char_ngrams(question) & eval_ngrams) / len(eval_ngrams)


def cap_by_topic(rows, cap, seed):
    """주제별 행 수가 cap을 넘으면 증강 행부터, 그래도 넘으면 원본 묶음째 줄인 행 리스트를 원래 순서대로 돌려줍니다(범위 밖 행은 모두 남김)."""
    # 주제별 행 수를 셉니다. 범위 밖 행은 상한 대상이 아니라 세지 않습니다
    topic_total = {}
    for row in rows:
        if row["source"] != "범위 밖":
            topic_total[row["source"]] = topic_total.get(row["source"], 0) + 1

    # 1단계: 증강 행의 id를 정렬한 뒤 난수 시드로 섞습니다
    topic_by_id = {}
    augmented_ids = []
    for row in rows:
        topic_by_id[row["id"]] = row["source"]
        if row["method"] == "패러프레이즈" or row["method"] == "역번역":
            augmented_ids.append(row["id"])
    augmented_ids = sorted(augmented_ids)
    random.Random(seed).shuffle(augmented_ids)

    # 섞은 순서대로 보며, 그 주제가 아직 cap을 넘으면 증강 행을 하나씩 뺍니다
    removed_ids = set()
    for row_id in augmented_ids:
        topic = topic_by_id[row_id]
        if topic_total[topic] > cap:
            removed_ids.add(row_id)
            topic_total[topic] = topic_total[topic] - 1

    # 2단계: 남은 행으로 원본 묶음마다 주제와 행 수를 셉니다
    bundle_topic = {}
    bundle_size = {}
    for row in rows:
        if row["source"] != "범위 밖" and row["id"] not in removed_ids:
            bundle_topic[row["origin_id"]] = row["source"]
            bundle_size[row["origin_id"]] = bundle_size.get(row["origin_id"], 0) + 1

    # 묶음 이름을 정렬해 섞은 순서대로 보며, 그래도 cap을 넘는 주제는 원본 묶음째 뺍니다
    origins = sorted(bundle_size)
    random.Random(seed).shuffle(origins)
    removed_origins = set()
    for origin in origins:
        topic = bundle_topic[origin]
        if topic_total[topic] > cap:
            removed_origins.add(origin)
            topic_total[topic] = topic_total[topic] - bundle_size[origin]

    # 빼지 않은 행을 원래 순서대로 모읍니다
    kept = []
    for row in rows:
        if row["id"] not in removed_ids and row["origin_id"] not in removed_origins:
            kept.append(row)
    return kept


def count_leaked(train, val):
    """검증 행 가운데 같은 origin_id가 학습 데이터에도 있는 행 수를 돌려줍니다."""
    train_origins = {row["origin_id"] for row in train}
    leaked = 0
    for row in val:
        if row["origin_id"] in train_origins:
            leaked += 1
    return leaked


def split_by_group(rows, key, val_ratio, seed):
    """key 값이 같은 행을 한 묶음으로 섞어 나눈 (학습 행 리스트, 검증 행 리스트)를 돌려줍니다."""
    # 묶음 이름을 정렬한 뒤 난수 시드로 섞습니다
    groups = sorted({row[key] for row in rows})  # set은 순서가 없어 정렬한 뒤 섞음
    random.Random(seed).shuffle(groups)

    # 앞쪽 묶음을 검증 묶음으로 씁니다
    val_groups = set(groups[:round(len(groups) * val_ratio)])  # 묶음 수의 val_ratio만큼이 검증 묶음

    train = []
    val = []
    for row in rows:
        if row[key] in val_groups:
            val.append(row)
        else:
            train.append(row)
    return train, val


# 난수 시드와 검증 비율(교안 02와 같은 값). 주제 상한, 행 단위 분할, 원본 단위 분할, 데이터 카드에 같은 값을 씁니다
SPLIT_SEED = 42
VAL_RATIO = 0.2

In [ ]:
# 고정 파일 198행과 평가 질문 16개
commerce_rows = load_jsonl(data_dir / "과제" / "commerce_qa_prepared.jsonl")
commerce_eval_questions = json.loads((data_dir / "과제" / "commerce_eval_questions.json").read_text(encoding="utf-8"))
print("행 수:", len(commerce_rows), "| 평가 질문 수:", len(commerce_eval_questions))

# 만든 방법별 행 수
commerce_table = pd.DataFrame(commerce_rows)
display(commerce_table["method"].value_counts())

In [ ]:
# 다음 단원에서 파인튜닝할 모델의 토크나이저. 처음 실행하면 토크나이저 파일만 내려받습니다
from transformers import AutoTokenizer

chat_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
print("끝 토큰:", chat_tokenizer.eos_token)

## 1. 표본 질문 증강하고 숫자 확인하기

**배경**: 쇼핑몰 이용자는 같은 내용을 여러 말투로 묻는데, 표현을 바꾸다 숫자가 바뀐 질문은 다른 것을 묻는 질문이라 원본과 숫자 리스트가 같은 행만 더합니다.

**요구사항**
- (1) 증강 대상은 근거 조문으로 답한 원본 Q&A인 문서 생성 행입니다. 범위 밖 행은 서로 다른 질문이 이미 같은 거절 답을 가져 표현을 더 늘릴 필요가 없으므로 증강하지 않고, 패러프레이즈·역번역·기존 FAQ 행은 원본이 아니라서 뺍니다. `commerce_rows` 가운데 `method`가 `"문서 생성"`인 행을 순서대로 **`grounded_rows`** 에 모읍니다.
- (2) `random.Random(42).sample(grounded_rows, 10)`으로 뽑은 10행을 **`augment_sample`** 에 담습니다.
- (3) 교안 02와 같은 스키마 `Paraphrase`(문자열 필드 `text`)와 셀에 적어 둔 `paraphrase_prompt`를 이어 체인 **`paraphrase_generator`** 를 만듭니다.
- (4) 같은 방법으로 `Translated`(문자열 필드 `text`)와 `translate_prompt`를 이어 체인 **`translator`** 를 만듭니다.
- (5) 표본 10개 질문을 질문마다 `{"text": 질문}` 입력으로 `paraphrase_generator.batch`에 한꺼번에 보내고(`config={"max_concurrency": MAX_CONCURRENCY}`, `return_exceptions=True`), 결과 리스트를 **`paraphrase_results`** 에 담습니다.
- (6) 성공한 결과마다 원본 행을 복사해 `id`에 `"-p"`를 붙이고 `method`를 `"패러프레이즈"`, `question`을 `result.text`로 바꾼 행을 **`paraphrase_rows`** 에 담습니다.
- (7) 표본의 앞 3개 질문을 `{"language": "영어", "text": 질문}`으로 `translator.batch`에 (5)와 같은 `config={"max_concurrency": MAX_CONCURRENCY}`, `return_exceptions=True`를 주어 보내고, 영어로 번역한 결과 리스트를 **`english_results`** 에 담습니다.
- (8) 영어 번역에 성공한 행과 영어 문장을 같은 순서로 모은 뒤(셀의 `english_rows`, `english_texts`), 영어 문장을 `{"language": "한국어", "text": 영어 문장}`으로 (5)와 같은 `config`, `return_exceptions=True`를 주어 다시 보내고, 한국어로 번역한 결과 리스트를 **`korean_results`** 에 담습니다.
- (9) 한국어 번역에 성공한 결과마다 원본 행을 복사해 `id`에 `"-b"`, `method`는 `"역번역"`, `question`은 `result.text`로 바꾼 행을 **`back_translation_rows`** 에 담습니다.
- (10) 두 리스트의 행마다 원본 질문(`origin_id`로 찾음)과 숫자 리스트(`re.findall(r"\d+", ...)`을 정렬한 리스트)을 비교해, 같은 행을 **`augmented_rows`** 에 담습니다.
- (11) 숫자 리스트가 다른 행은 키가 `id`, `원본 질문`, `증강 질문` 순서인 딕셔너리로 **`augment_dropped`** 에 담습니다.
- 이 문항의 결과는 실행마다 달라지므로 뒤 문항에 섞지 않습니다. 2번부터는 고정 파일 `commerce_rows`만 씁니다.

**예시**: "5만 원 결제했는데 품절이라고 하면 환불은 언제 받나요?"(숫자 `['5']`)를 "5만 원을 냈는데 재고가 없으면 언제 환불되나요?"로 바꾸면 남고, "50만 원을 …"으로 바꾸면 숫자 리스트가 `['50']`이 되어 `augment_dropped`로 갑니다.

**확인 기준**
- `grounded_rows`, `augment_sample`: `grounded_rows`가 문서 생성 행만이고(범위 밖 행 없음), `augment_sample`의 id가 자가채점 셀에 적어 둔 10개와 같습니다.
- `paraphrase_generator`, `translator`: 셀의 프롬프트로 시작하고 `Paraphrase`, `Translated`를 돌려줍니다.
- `paraphrase_results`, `english_results`, `korean_results`: 성공한 패러프레이즈가 하나 이상이고, `korean_results`가 영어 번역에 성공한 문장마다 하나씩입니다.
- `paraphrase_rows`, `back_translation_rows`: 성공한 결과마다 한 행씩 있고, 질문이 그 결과의 `text`입니다. 표본 행(`augment_sample`)과 `commerce_rows`는 바뀌지 않은 고정 파일 그대로입니다.
  - 생성용 LLM이 원본과 같은 문장을 돌려주면 패러프레이즈 질문이 원본 질문과 같을 수 있습니다. 이런 행도 그대로 두며, 답까지 원본과 같아 교안 02처럼 데이터셋에 합치면 정확 중복 제거에서 빠집니다.
- `augmented_rows`: 행마다 키 순서가 원본 행과 같고, `id`가 원본 `id` + `"-p"`(`method` `"패러프레이즈"`) 또는 + `"-b"`(`method` `"역번역"`, 표본 앞 3개만)입니다. `origin_id`와 `answer`가 원본과 같고 숫자 리스트가 원본 질문과 같습니다.
- `augment_dropped`: 딕셔너리의 키가 위 순서이고, 행마다 숫자 리스트가 원본과 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 패러프레이즈, 역번역, 뜻 보존 확인과 같은 순서를 표본 10개에만 적용합니다.

세부구현:
1. 스키마는 필드 하나짜리 Pydantic 모델이고, 체인은 프롬프트 템플릿과 구조화 출력 llm을 |로 이은 것입니다.
2. 새 행은 dict(원본 행)으로 복사한 뒤 바뀌는 키만 고칩니다. 원본 행을 직접 고치면 고정 파일의 행이 바뀝니다.
3. 역번역의 두 번째 .batch는 영어 번역에 성공한 행만 받으므로, 성공한 행과 영어 문장을 같은 순서의 리스트 두 개로 모아 다시 짝짓습니다.
```

</details>

In [ ]:
# [작성] (1), (2) 증강 대상인 문서 생성 행에서 표본 10개를 뽑으세요.

# 범위 밖 행과 이미 늘린 행, 기존 FAQ는 빼고 문서 생성 행만 순서대로 모은 뒤 시드 42로 10개 뽑기
grounded_rows = []
for row in commerce_rows:
    ...
augment_sample = ...
print("표본 id:", [row["id"] for row in augment_sample])

# 증강할 표본 10행(생성용 LLM에 보낼 원본 질문 question 포함)
display(pd.DataFrame(augment_sample))

# 원본 질문을 id로 찾는 딕셔너리(증강 질문과 나란히 볼 때 씁니다)
sample_question_by_id = {row["id"]: row["question"] for row in augment_sample}

In [ ]:
# [작성] (3) 바꿔 쓴 문장 하나를 받는 패러프레이즈 체인을 만드세요.
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class Paraphrase(BaseModel):
    """같은 뜻을 다른 표현으로 쓴 문장 하나입니다."""
    # text 문자열 필드 하나를 Field와 description으로 쓰세요
    ...


# 숫자, 기간, 이름이 바뀌면 다른 것을 묻는 질문이 되므로 그대로 두라고 합니다
paraphrase_prompt = ChatPromptTemplate.from_messages([
    ("system", "온라인 쇼핑 상담 챗봇 학습 데이터의 질문을 뜻은 그대로 두고 다른 표현으로 바꿔 써야 한다. 바꿔 쓴 질문에는 원래 질문의 답이 그대로 붙는다."),
    ("human", """아래 [문장]을 뜻은 그대로 두고 표현(어순, 낱말, 말투)만 바꿔 한 문장으로 다시 써라.

뜻 지키기:
1. 숫자, 기간, 이름은 원문 그대로 둔다.
2. 새로운 정보를 더하거나 빼지 않는다.

말투:
3. 온라인 쇼핑 이용자나 판매자가 실제로 물어볼 말투로 쓴다.

[문장]
{text}"""),
])

# 프롬프트 -> 생성용 LLM(Paraphrase 모양으로만 답함) 체인
paraphrase_generator = ...

In [ ]:
# [작성] (4) 번역문 하나를 받는 번역 체인을 만드세요(영어 번역과 한국어 번역에 함께 씁니다).


class Translated(BaseModel):
    """번역 결과입니다."""
    # text 문자열 필드 하나를 Field와 description으로 쓰세요
    ...


# 숫자가 바뀌지 않게 하고, 설명 없이 번역문만 받습니다
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문장을 자연스럽게 번역해야 한다."),
    ("human", """아래 [문장]을 {language}로 자연스럽게 번역해라. 숫자는 그대로 두고 번역문만 써라.

[문장]
{text}"""),
])

# 프롬프트 -> 생성용 LLM(Translated 모양으로만 답함) 체인
translator = ...

In [ ]:
# [작성] (5), (6) 표본 질문 10개를 패러프레이즈해 새 행을 만드세요.

# .batch로 동시에 처리할 요청 수의 상한. 429 오류(요청 한도 초과)가 나면 줄입니다
MAX_CONCURRENCY = 32

# 표본 질문마다 입력 딕셔너리를 만들어 한꺼번에 보내기(실패한 요청은 예외 객체로)
paraphrase_inputs = ...
paraphrase_results = ...

# 성공한 결과마다 원본 행을 복사해 id, method, question만 바꾸기
paraphrase_rows = []
for row, result in zip(augment_sample, paraphrase_results):
    ...

print("패러프레이즈 행 수:", len(paraphrase_rows))

# 바꾼 질문(question) 옆에 원본 질문을 붙여 봅니다
paraphrase_table = pd.DataFrame(paraphrase_rows)
paraphrase_table["원본 질문"] = [sample_question_by_id[row["origin_id"]] for row in paraphrase_rows]
display(paraphrase_table)

In [ ]:
# [작성] (7)~(9) 표본 앞 3개 질문을 영어로 번역했다가 다시 한국어로 번역해 새 행을 만드세요.
back_sample = augment_sample[:3]  # 질문 하나에 호출이 두 번이라 앞 3개만

# 1차: 영어로 번역
english_inputs = ...
english_results = ...

# 영어 번역에 성공한 행과 영어 문장만 같은 순서로 모으기
english_rows = []
english_texts = []
for row, result in zip(back_sample, english_results):
    ...

# 2차: 영어 문장을 다시 한국어로 번역
korean_inputs = ...
korean_results = ...

# 성공한 결과마다 원본 행을 복사해 id, method, question만 바꾸기
back_translation_rows = []
for row, result in zip(english_rows, korean_results):
    ...

print("영어 번역 성공 수:", len(english_rows), "| 역번역 행 수:", len(back_translation_rows))

# 역번역 질문(question) 옆에 원본 질문과 중간의 영어 문장을 붙여 봅니다
english_by_id = {row["origin_id"]: text for row, text in zip(english_rows, english_texts)}
back_translation_table = pd.DataFrame(back_translation_rows)
back_translation_table["원본 질문"] = [sample_question_by_id[row["origin_id"]] for row in back_translation_rows]
back_translation_table["영어"] = [english_by_id[row["origin_id"]] for row in back_translation_rows]
display(back_translation_table)

In [ ]:
# [작성] (10), (11) 원본 질문과 숫자 리스트가 같은 증강 행만 남기세요.
import re

# 원본 질문을 id로 찾는 딕셔너리
question_by_id = {row["id"]: row["question"] for row in commerce_rows}

augmented_rows = []
augment_dropped = []
for row in paraphrase_rows + back_translation_rows:
    ...

print("통과한 증강 행 수:", len(augmented_rows), "| 숫자가 달라 걸린 행 수:", len(augment_dropped))
display(pd.DataFrame(augmented_rows))
display(pd.DataFrame(augment_dropped, columns=["id", "원본 질문", "증강 질문"]))

In [ ]:
# [자가채점] 표본, 두 체인, 증강 행의 불변식을 확인합니다.
from langchain_core.runnables import RunnableSequence

expected_sample_ids = ["c041", "c008", "c002", "c048", "c018", "c016", "c015", "c009", "c058", "c007"]

# 표본 행은 고정 파일의 원본 문서 생성 행 그대로여야 합니다(증강 행을 복사하지 않고 고치면 표본 행의 id와 method가 바뀝니다)
for row in augment_sample:
    assert not row["id"].endswith(("-p", "-b")), f"표본 행의 id가 {row['id']}, method가 {row['method']}로 바뀌었습니다. 원본 행을 dict(row)로 복사한 뒤 고치고 준비 셀부터 다시 실행하세요."
for row in grounded_rows:
    assert row["method"] == "문서 생성", f"grounded_rows에 {row['method']} 행({row['id']})이 있습니다. 문서 생성 행만 모으세요. 범위 밖 행은 증강하지 않습니다."
for row in augment_sample:
    assert row["method"] == "문서 생성", f"표본 행 {row['id']}의 method가 {row['method']}입니다. 원본 행을 dict(row)로 복사한 뒤 고치고 준비 셀부터 다시 실행하세요."
sample_ids = []
for row in augment_sample:
    sample_ids.append(row["id"])
assert sample_ids == expected_sample_ids, f"표본 id가 {sample_ids}입니다. 문서 생성 행을 파일 순서대로 grounded_rows에 모았는지, 지문의 시드 42로 만든 난수 생성기로 10개를 뽑았는지 확인하세요."

# 두 체인: 셀의 프롬프트로 시작하고 스키마 모양으로 돌려줌
assert isinstance(paraphrase_generator, RunnableSequence) and paraphrase_generator.first is paraphrase_prompt, "paraphrase_generator는 paraphrase_prompt로 시작하는 체인입니다."
assert paraphrase_generator.output_schema.model_json_schema()["title"] == "Paraphrase", "paraphrase_generator가 Paraphrase를 돌려주지 않습니다. llm에 Paraphrase로 구조화 출력을 거세요."
assert isinstance(translator, RunnableSequence) and translator.first is translate_prompt, "translator는 translate_prompt로 시작하는 체인입니다."
assert translator.output_schema.model_json_schema()["title"] == "Translated", "translator가 Translated를 돌려주지 않습니다. llm에 Translated로 구조화 출력을 거세요."
assert list(paraphrase_generator.output_schema.model_json_schema()["properties"]) == ["text"], "Paraphrase는 문자열 필드 text 하나입니다."
assert list(translator.output_schema.model_json_schema()["properties"]) == ["text"], "Translated는 문자열 필드 text 하나입니다."

# 증강 행: 원본 행을 복사해 id, method, question만 바뀜
original_by_origin = {row["origin_id"]: row for row in augment_sample}
paraphrase_count = 0
back_count = 0
for new_row in augmented_rows:
    assert new_row["origin_id"] in original_by_origin, f"{new_row['id']}의 origin_id가 표본 행과 다릅니다. 원본 행을 복사해 origin_id를 그대로 두세요."
    original = original_by_origin[new_row["origin_id"]]
    assert list(new_row) == list(original), f"{new_row['id']}의 키가 원본 행과 다릅니다. dict(원본 행)으로 복사한 뒤 값만 바꾸세요."
    assert new_row["answer"] == original["answer"], f"{new_row['id']}의 답이 원본과 다릅니다. 증강은 질문만 바꿉니다."
    if new_row["id"] == original["id"] + "-p":
        assert new_row["method"] == "패러프레이즈", f"{new_row['id']}의 method는 \"패러프레이즈\"입니다."
        paraphrase_count += 1
    else:
        assert new_row["id"] == original["id"] + "-b", f"{new_row['id']}: 증강 행 id는 원본 id 뒤에 -p 또는 -b를 붙입니다."
        assert new_row["method"] == "역번역", f"{new_row['id']}의 method는 \"역번역\"입니다."
        assert original in augment_sample[:3], f"{new_row['id']}: 역번역은 표본 앞 3개에만 합니다."
        back_count += 1
    assert sorted(re.findall(r"\d+", new_row["question"])) == sorted(re.findall(r"\d+", original["question"])), f"{new_row['id']}는 숫자 리스트가 원본 질문과 다른데 augmented_rows에 있습니다."
assert paraphrase_count <= 10 and back_count <= 3, "패러프레이즈 행은 10개 이하, 역번역 행은 3개 이하입니다."

# 걸린 행: 숫자 리스트가 원본과 다름
for dropped in augment_dropped:
    assert list(dropped) == ["id", "원본 질문", "증강 질문"], "augment_dropped의 딕셔너리는 id, 원본 질문, 증강 질문 세 키입니다."
    assert sorted(re.findall(r"\d+", dropped["원본 질문"])) != sorted(re.findall(r"\d+", dropped["증강 질문"])), f"{dropped['id']}는 숫자 리스트가 같은데 걸렸습니다."
assert len(augmented_rows) + len(augment_dropped) == len(paraphrase_rows) + len(back_translation_rows), "증강 행마다 augmented_rows나 augment_dropped 한쪽에 넣으세요."

# 만든 증강 행 수: 성공한 결과마다 한 행이고, 질문은 그 결과의 text입니다
paraphrase_texts = []
for result in paraphrase_results:
    if not isinstance(result, Exception):
        paraphrase_texts.append(result.text)
assert len(paraphrase_texts) >= 1, "패러프레이즈 요청이 모두 실패했습니다. 오류 메시지(paraphrase_results의 예외 객체)를 보고 (5), (6) 셀을 다시 실행하세요."
assert len(paraphrase_rows) == len(paraphrase_texts), f"paraphrase_rows가 {len(paraphrase_rows)}행인데 성공한 패러프레이즈 결과는 {len(paraphrase_texts)}개입니다. 성공한 결과마다 원본 행을 복사한 새 행을 하나씩 더하세요."
for new_row, text in zip(paraphrase_rows, paraphrase_texts):
    assert new_row["question"] == text, f"{new_row['id']}의 질문이 패러프레이즈 결과와 다릅니다. question에 result.text를 넣었는지 확인하세요."
english_success = 0
for result in english_results:
    if not isinstance(result, Exception):
        english_success += 1
assert len(korean_results) == english_success, f"korean_results가 {len(korean_results)}개인데 성공한 영어 번역은 {english_success}개입니다. 영어 번역에 성공한 행과 영어 문장을 모아 한국어 번역으로 보냈는지 확인하세요."
korean_texts = []
for result in korean_results:
    if not isinstance(result, Exception):
        korean_texts.append(result.text)
assert len(back_translation_rows) == len(korean_texts), f"back_translation_rows가 {len(back_translation_rows)}행인데 성공한 한국어 번역은 {len(korean_texts)}개입니다. 성공한 결과마다 원본 행을 복사한 새 행을 하나씩 더하세요."
for new_row, text in zip(back_translation_rows, korean_texts):
    assert new_row["question"] == text, f"{new_row['id']}의 질문이 한국어 번역 결과와 다릅니다. question에 result.text를 넣었는지 확인하세요."
assert len(paraphrase_rows) + len(back_translation_rows) >= 1, "만든 증강 행이 없습니다. 성공한 결과마다 원본 행을 복사해 새 행을 만드세요."

# 고정 파일은 행 하나하나까지 그대로여야 합니다
assert commerce_rows == load_jsonl(data_dir / "과제" / "commerce_qa_prepared.jsonl"), "commerce_rows가 고정 파일과 다릅니다. 원본 행을 직접 고치거나 증강 행을 commerce_rows에 더하지 말고, dict(row)로 복사한 뒤 고치고 준비 셀부터 다시 실행하세요."
print("✅ 통과!")

**결과 읽기**
- **걸린 행:** 표본에는 숫자가 든 질문이 다섯 개 있습니다. "5만 원"을 "50000원"으로 쓴 것처럼 같은 값을 다르게 적은 질문도 걸리므로 원본과 나란히 읽어 판정합니다.
- **통과한 행:** "언제"가 "얼마"로 바뀐 것처럼 숫자 없이 뜻이 바뀐 질문은 숫자 비교로 잡지 못하므로 원본과 나란히 읽습니다.

## 2. 품질 필터 네 단계 적용하고 버린 행 판독하기

**배경**: 문서 생성 행, 증강 행, 기존 FAQ를 합친 고정 파일에는 번역 결과가 빈 문자열로 온 역번역 행, 질문이나 답이 비었거나 답이 한 단어뿐인 기존 FAQ 행, 같은 행이 그대로 또는 띄어쓰기와 끝맺음만 다르게 한 번 더 들어간 중복 행이 섞여 있어, 교안 02의 네 단계를 같은 순서로 적용하고 어떤 결함이 어디서 들어왔는지 알 수 있게 단계마다 버린 행의 id와 만든 방법(`method`)도 읽습니다.

**요구사항**
- (1) `commerce_rows` 가운데 `format_ok`를 통과한 행만 순서대로 **`after_format`** 에 담습니다.
- (2) `after_format` 가운데 `length_ok`(기본 `min_a=5`)를 통과한 행만 **`after_len`** 에 담습니다.
- (3) `after_len`에 `dedup_exact`를 적용한 결과를 **`after_dedup`** 에 담습니다.
- (4) `after_dedup`에 `dedup_similar`(기본 기준 0.8)를 적용한 결과를 **`clean`** 에 담습니다.
- (5) 단계마다 남은 행 수와 그 단계에서 뺀 행 수를 데이터프레임(DataFrame) **`filter_counts`**(열 `"단계"`, `"남은 수"`, `"제거 수"`)로 만듭니다. 행은 `"합친 데이터"`(제거 수 0), `"형식 검증"`, `"길이 필터"`, `"정확 중복 제거"`, `"유사 중복 제거"` 순서입니다.
- (6) 단계 이름(`"형식 검증"`, `"길이 필터"`, `"정확 중복 제거"`, `"유사 중복 제거"`)을 키로, 그 단계에서 버린 행의 `id` 리스트(파일 순서)를 값으로 하는 딕셔너리 **`dropped_ids`** 를 `dropped_rows`로 만듭니다. 셀 끝의 제공 코드가 버린 행을 단계와 함께 테이블로 보여 줍니다.

**확인 기준**
- 네 리스트와 `filter_counts`: 네 리스트의 행 수가 자가채점 셀에 적어 둔 기대값과 같고, 각 리스트는 앞 단계 리스트에서 행을 순서대로 고른 리스트입니다. `filter_counts`가 위 다섯 행, 세 열이고 남은 수와 제거 수가 기대값과 같습니다.
- `dropped_ids`: 키가 위 네 단계 이름이고, 단계마다 id 리스트가 자가채점 셀에 적어 둔 기대값과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 앞 두 단계는 반복문으로 통과한 행만 담고, 뒤 두 단계는 리스트를 받아 남긴 리스트를 돌려주는 함수를 부릅니다.

세부구현:
1. 테이블은 단계마다 딕셔너리 한 줄씩 다섯 줄의 리스트로 만들어 데이터프레임에 넣습니다.
2. 제거 수는 바로 앞 단계의 남은 수에서 이 단계의 남은 수를 뺀 값입니다.
3. 버린 행은 단계마다 그 단계의 입력 리스트와 결과 리스트를 dropped_rows에 넣어 받습니다. 셀의 stages 리스트가 단계 이름, 입력, 결과를 한 줄씩 묶어 둡니다.
```

</details>

In [ ]:
# [작성] (1)~(4) 품질 필터 네 단계를 차례로 적용하고, (5) 단계별 테이블을 만드세요.

# (1) 형식 검증
after_format = []
for row in commerce_rows:
    ...

# (2) 길이 필터
after_len = []
for row in after_format:
    ...

# (3) 정확 중복 제거와 (4) 유사 중복 제거
after_dedup = ...
clean = ...

# (5) 단계마다 남은 수와 그 단계에서 뺀 수
filter_counts = ...
display(filter_counts)

In [ ]:
# [작성] (6) 단계마다 버린 행의 id를 모으세요.

# (단계 이름, 그 단계의 입력, 그 단계의 결과)
stages = [
    ("형식 검증", commerce_rows, after_format),
    ("길이 필터", after_format, after_len),
    ("정확 중복 제거", after_len, after_dedup),
    ("유사 중복 제거", after_dedup, clean),
]

dropped_ids = {}
dropped_table_rows = []
for stage_name, before, after in stages:
    # 이 단계에서 버린 행과 그 id 리스트
    stage_rows = ...
    dropped_ids[stage_name] = ...

    # 판독용 테이블에 넣을 행: 단계 이름을 맨 앞에 두고 원래 행의 키를 모두 붙입니다
    for row in stage_rows:
        table_row = {"단계": stage_name}
        table_row.update(row)
        dropped_table_rows.append(table_row)

# 버린 행의 모든 열(answer 키가 없는 행은 빈 칸 NaN)
dropped_table = pd.DataFrame(dropped_table_rows)
display(dropped_table)

In [ ]:
# [자가채점] 단계별 남은 수, 단계 사이의 순서, 단계별 테이블, 버린 행의 id를 확인합니다.
expected_counts = [198, 194, 191, 187, 183]
expected_removed = [0, 4, 3, 4, 4]
expected_dropped_ids = {"형식 검증": ["c016-b", "f05", "f13", "f19"], "길이 필터": ["f03", "f10", "f15"], "정확 중복 제거": ["x01", "x03", "x06", "x09"], "유사 중복 제거": ["x02", "x04", "x08", "x10"]}
stage_names = ["합친 데이터", "형식 검증", "길이 필터", "정확 중복 제거", "유사 중복 제거"]

actual_counts = [len(commerce_rows), len(after_format), len(after_len), len(after_dedup), len(clean)]
assert actual_counts == expected_counts, f"단계별 남은 수가 {actual_counts}입니다. 기대값은 {expected_counts}입니다. 단계 순서와 쓰는 함수를 확인하세요."

# 각 단계의 결과는 앞 단계 리스트에서 행을 순서대로 고른 리스트입니다
for before, after in [(commerce_rows, after_format), (after_format, after_len), (after_len, after_dedup), (after_dedup, clean)]:
    position = 0
    for row in before:
        if position < len(after) and row == after[position]:
            position += 1
    assert position == len(after), "각 단계의 결과는 바로 앞 단계 리스트에서 행을 순서대로 골라 담은 리스트입니다. 앞 단계의 결과를 입력으로 썼는지 확인하세요."

# 단계별 테이블
assert isinstance(filter_counts, pd.DataFrame), "filter_counts는 데이터프레임입니다."
assert list(filter_counts.columns) == ["단계", "남은 수", "제거 수"], "filter_counts의 열은 단계, 남은 수, 제거 수입니다."
assert list(filter_counts["단계"]) == stage_names, f"단계 행은 {stage_names} 순서입니다."
assert list(filter_counts["남은 수"]) == expected_counts, "남은 수 열이 단계별 행 수와 다릅니다."
assert list(filter_counts["제거 수"]) == expected_removed, "제거 수는 바로 앞 단계의 남은 수에서 이 단계의 남은 수를 뺀 값입니다(합친 데이터는 0)."

# 단계마다 버린 행의 id(파일 순서)
assert isinstance(dropped_ids, dict) and list(dropped_ids) == list(expected_dropped_ids), f"dropped_ids의 키는 {list(expected_dropped_ids)} 네 단계 이름입니다."
for name in expected_dropped_ids:
    assert dropped_ids[name] == expected_dropped_ids[name], f"{name}에서 버린 id가 {dropped_ids[name]}입니다. dropped_rows(그 단계의 입력, 그 단계의 결과)로 버린 행을 받아 id만 파일 순서대로 모으세요."
print("✅ 통과!")

**결과 읽기**
- **형식과 길이:** `c016-b`·`f05`는 빈 질문(`c016-b`는 번역이 빈 문자열로 온 역번역 행), `f13`·`f19`은 `answer` 키가 없는 기존 FAQ 행입니다. 길이 필터 `f03`·`f10`·`f15`는 "가능."처럼 조건이 빠진 짧은 기존 FAQ 답입니다.
- **중복:** 정확 중복 `x01`·`x03`·`x06`·`x09`는 `c019`, `c037-b`, `c047-p`, `f17`을 그대로 옮긴 행이고, 유사 중복 `x02`·`x04`·`x08`·`x10`는 `c023`, `c040-p`, `f12`, `c054-b`에서 띄어쓰기나 끝맺음만 바꾼 행입니다.
- **통과한 행:** 평가 질문을 옮긴 행은 행끼리 비교하는 네 단계에 걸리지 않아 3번 오염 검사에서 거릅니다.

## 3. 평가 질문과 겹치는 원본 묶음 빼기

**배경**: 평가 질문과 같거나 거의 같은 질문을 학습하면 평가 점수가 실제 실력보다 높게 나오므로, 품질 필터를 거친 `clean`에서 그런 행의 원본 묶음을 주제 상한과 분할보다 먼저 뺍니다.

**요구사항**
- (1) `clean`의 행마다 평가 질문 16개와의 `ngram_overlap(질문, 평가 질문)` 가운데 가장 큰 값과 그 평가 질문을 구합니다. 가장 큰 값이 `CONTAMINATION_THRESHOLD`(0.7) 이상인 행을 키가 `"id"`, `"origin_id"`, `"method"`, `"question"`, `"평가 질문"`, `"겹침"` 순서인 딕셔너리로 **`contaminated`** 에 담습니다.
- (2) 걸린 행의 `origin_id` 집합 **`contaminated_origins`** 를 만듭니다.
- (3) `clean`에서 `origin_id`가 `contaminated_origins`에 있는 행을 뺀 리스트 **`decontaminated`** 를 만듭니다. 4번부터는 `decontaminated`를 씁니다.

**예시**: 평가 질문 "주문하고 나서 며칠 안에 상품 발송 준비를 마쳐야 하나요?"와 끝맺음만 다른 "… 마쳐야 하는지요?"는 3-gram 겹침이 0.86입니다. 이 행이 걸리면 같은 `origin_id`의 원본과 역번역도 함께 뺍니다.

**확인 기준**
- `contaminated`: 딕셔너리마다 위 여섯 키이고, 겹침이 그 행의 질문과 기록한 평가 질문의 `ngram_overlap`이자 평가 질문 가운데 가장 큰 값이며 0.7 이상입니다. id 리스트가 자가채점 셀의 기대값과 같습니다.
- `contaminated_origins`: 걸린 행의 `origin_id` 집합입니다.
- `decontaminated`: `clean`에서 행을 순서대로 고른 리스트이고 행 수가 기대값과 같으며, 어느 행의 `origin_id`도 `contaminated_origins`에 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 오염 검사와 같은 순서입니다. 행마다 평가 질문 전체를 돌며 가장 큰 겹침을 기록하고, 기준 이상인 행을 모은 뒤 그 원본 묶음을 뺍니다.

세부구현:
1. 가장 큰 값을 찾을 때는 0과 빈 문자열에서 시작해, 더 큰 겹침을 만나면 값과 평가 질문을 함께 바꿉니다.
2. 원본 묶음을 뺄 때는 걸린 행이 아니라 origin_id로 비교합니다.
```

</details>

In [ ]:
# [작성] (1) 평가 질문과 겹치는 행을 찾고, (2), (3) 그 원본 묶음을 뺀 리스트를 만드세요.

# 오염 기준(교안 02와 같은 값)
CONTAMINATION_THRESHOLD = 0.7

# (1) 행마다 가장 많이 겹치는 평가 질문과 그 겹침
contaminated = []
for row in clean:
    best_overlap = 0
    best_eval = ""
    ...

contaminated_table = pd.DataFrame(contaminated, columns=["id", "origin_id", "method", "question", "평가 질문", "겹침"])
display(contaminated_table.round(2))  # 겹침은 소수 둘째 자리까지

# (2) 걸린 원본 묶음과 (3) 그 묶음을 뺀 행
contaminated_origins = ...
decontaminated = []
for row in clean:
    ...
print("품질 필터 뒤:", len(clean), "행 -> 오염 묶음을 뺀 뒤:", len(decontaminated), "행")

In [ ]:
# [자가채점] 걸린 행, 원본 묶음, 오염 묶음을 뺀 행을 확인합니다.
expected_contaminated_ids = ["c012-p", "x05", "x07", "x11"]

contaminated_ids = []
for row in contaminated:
    assert list(row) == ["id", "origin_id", "method", "question", "평가 질문", "겹침"], "contaminated의 딕셔너리는 id, origin_id, method, question, 평가 질문, 겹침 여섯 키입니다."
    assert row["겹침"] >= 0.7, f"{row['id']}의 겹침({row['겹침']})이 기준 0.7보다 작습니다. 기준 이상인 행만 담으세요."
    assert row["평가 질문"] in commerce_eval_questions, f"{row['id']}의 평가 질문이 commerce_eval_questions에 없습니다. 가장 많이 겹친 평가 질문을 기록하세요."

    # 겹침은 행의 질문과 기록한 평가 질문의 ngram_overlap이고, 평가 질문 전체 가운데 가장 큰 값입니다
    assert abs(row["겹침"] - ngram_overlap(row["question"], row["평가 질문"])) < 0.01, f"{row['id']}의 겹침({row['겹침']})이 ngram_overlap(행의 질문, 기록한 평가 질문)과 다릅니다. 인자 순서가 (질문, 평가 질문)인지, 겹침이 커질 때마다 그 값과 평가 질문을 함께 바꿨는지 확인하세요."
    for eval_question in commerce_eval_questions:
        assert ngram_overlap(row["question"], eval_question) <= row["겹침"] + 0.01, f"{row['id']}에 기록한 평가 질문보다 더 많이 겹치는 평가 질문이 있습니다. 평가 질문 전체를 돌며 가장 큰 겹침과 그 평가 질문을 함께 기록하세요."
    contaminated_ids.append(row["id"])
assert sorted(contaminated_ids) == sorted(expected_contaminated_ids), f"걸린 행이 {contaminated_ids}입니다. clean의 행마다 평가 질문 전체와의 겹침 가운데 가장 큰 값을 기준 0.7과 비교했는지 확인하세요."

# 원본 묶음과 남은 행(미리 구한 값)
origin_set = set()
for row in contaminated:
    origin_set.add(row["origin_id"])
assert contaminated_origins == origin_set, "contaminated_origins는 걸린 행의 origin_id 집합입니다."
position = 0
for row in clean:
    if position < len(decontaminated) and row == decontaminated[position]:
        position += 1
assert position == len(decontaminated), "decontaminated는 clean에서 행을 순서대로 고른 리스트입니다."
for row in decontaminated:
    assert row["origin_id"] not in contaminated_origins, f"{row['id']}는 걸린 원본 묶음의 행인데 남아 있습니다. 걸린 행만 빼지 말고 같은 origin_id의 행을 모두 빼세요."
assert len(decontaminated) == 177, f"decontaminated가 {len(decontaminated)}행입니다. clean에서 걸린 원본 묶음의 행을 모두 뺐는지 확인하세요."
print("✅ 통과!")

**결과 읽기**
- **걸린 4행:** `x05`, `x07`, `x11`은 평가 질문을 옮긴 기존 FAQ 행이고, `c012-p`는 평가 질문과 끝맺음만 다른 패러프레이즈입니다.
- **6행이 빠진 이유:** `c012-p`의 원본 묶음인 `c012`, `c012-b`까지 함께 빠졌습니다.
- **남는 한계:** 표현이 다른 같은 질문은 글자 겹침이 낮아 걸리지 않으므로, 평가 질문 리스트와 사람이 한 번 대조합니다.

## 4. 구성을 점검하고 주제 상한으로 균형 맞추기

**배경**: 품질 필터와 오염 검사는 행 하나하나만 보므로, 학습 데이터 전체의 구성(어떤 방법, 어떤 주제, 같은 답이 몇 번)을 테이블로 보고 행이 많은 주제를 줄여 모델이 많은 쪽에 맞춰지지 않게 합니다.

**요구사항**
- (1) `decontaminated`를 바꾼 데이터프레임 `decontaminated_table`(셀에 있음)에서 방법(`method`)별 행 수와 비율을 데이터프레임 **`method_table`**(행 이름은 방법, 열 `"행 수"`, `"비율(%)"`)로 만듭니다. 비율은 행 수를 `len(decontaminated)`로 나누고 100을 곱해 소수 첫째 자리로 반올림합니다. 범위 밖 행의 비율도 이 테이블에서 봅니다.
- (2) 교안 02의 구성 점검과 같은 방법으로 주제별 테이블 **`topic_table`**(열 `"행 수"`, `"문단 수"`, `"문단당 행 수"`, 행 이름은 주제 태그)을 만듭니다. 범위 밖 행을 뺀 행을 주제(`source`)마다 묶은 `topic_groups`(셀에 있음)에서 행 수는 묶음의 행 수, 문단 수는 `doc_id` 고유 수입니다. 문단당 행 수는 `doc_id`가 있는 행 수를 문단 수로 나눠 소수 첫째 자리로 반올림합니다(기존 FAQ 행은 `doc_id`가 빈 칸이라 분자에 들어가지 않습니다).
- (3) `topic_table`의 `"행 수"` 열을 주제별 행 수 **`topic_counts`**(Series)로 꺼냅니다. (5)의 주제 상한에 씁니다.
- (4) 범위 밖 행도 포함한 `decontaminated_table`의 `origin_id` 열로 원본 묶음마다 행 수(묶음 크기)를 센 `bundle_sizes`를 한 번 더 세어, 묶음 크기별 묶음 수를 크기 순서로 담은 **`size_counts`**(Series)를 만듭니다. 같은 원본 묶음의 행은 답이 같아, 묶음 크기가 같은 답이 몇 번 나오는지를 보여 줍니다.
- (5) `topic_counts`의 중앙값에 1.5를 곱해 `round`로 반올림한 주제 상한 **`TOPIC_CAP`** 을 구합니다.
- (6) 제공 함수 `cap_by_topic`에 `decontaminated`, `TOPIC_CAP`, `SPLIT_SEED`를 넣어 줄인 행 리스트를 **`balanced`** 에 담습니다. 상한을 넘는 주제는 증강 행부터 빼고, 그래도 넘으면 원본 묶음째 뺍니다. 범위 밖 행은 상한 대상이 아닙니다.
- 제공 셀이 주제별 행 수를 균형 전후로 나란히 놓은 테이블 `balance_table`을 보여 줍니다. 5번부터는 `balanced`를 씁니다.

**예시**: 주제별 행 수가 60, 40, 30, 20, 10이면 중앙값 30의 1.5배인 45가 상한입니다. 60행 주제는 섞은 순서대로 증강 행 15행을 빼 45행이 되고, 나머지 주제와 범위 밖 행은 그대로 남습니다.

**확인 기준**
- 구성 테이블: `method_table`, `topic_table`, `size_counts`의 값이 자가채점 셀에 적어 둔 기대값과 같고, `topic_table`이 행 수가 많은 순서이며 `topic_counts`에 "범위 밖"이 없습니다.
- `TOPIC_CAP`, `balanced`: 상한, 남은 행 수, 빠진 행의 id가 기대값과 같고, `balanced`는 `decontaminated`에서 행을 순서대로 고른 리스트입니다.
- 균형의 성질: 범위 밖 행은 모두 남고, 주제마다 행 수가 상한 이하입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 구성 점검과 주제 상한과 같은 순서입니다. 데이터프레임과 주제별 묶음에서 열마다 세고, 상한을 넘는 주제 줄이기는 제공 함수에 맡깁니다.

세부구현:
1. 비율은 행 수 Series를 전체 행 수로 나눈 Series에 100을 곱하고 round(1)로 반올림합니다. Series 여러 개를 딕셔너리 하나로 묶어 데이터프레임에 넣으면 같은 행 이름끼리 나란히 열이 됩니다.
2. 주제별 묶음의 행 수는 size, doc_id 고유 수는 nunique, doc_id가 있는 행 수는 count로 셉니다. nunique와 count는 빈 칸(NaN)을 세지 않습니다.
3. 묶음 크기별 묶음 수는 origin_id 열의 value_counts 결과에 value_counts를 한 번 더 부르고, 행 이름(크기) 순서로 정렬합니다.
4. Series의 median()이 중앙값입니다.
```

</details>

In [ ]:
# [작성] (1) 방법별 행 수와 비율(%)을 테이블로 만드세요.
decontaminated_table = pd.DataFrame(decontaminated)
method_counts = decontaminated_table["method"].value_counts()
method_table = ...
display(method_table)

In [ ]:
# [작성] (2) 주제별 행 수, 문단 수, 문단당 행 수를 테이블로 만들고, (3) 주제별 행 수를 꺼내세요.

# 범위 밖 행은 주제가 없어 빼고, 주제마다 묶습니다
topic_rows_table = decontaminated_table[decontaminated_table["method"] != "범위 밖"]
topic_groups = topic_rows_table.groupby("source")

# 행 수와 문단 수(doc_id 고유 수), 문단당 행 수(doc_id가 있는 행 수 / 문단 수)
topic_table = ...
topic_table["문단당 행 수"] = ...
topic_table = topic_table.sort_values("행 수", ascending=False)

# (3) 주제별 행 수는 뒤의 주제 상한에서도 씁니다
topic_counts = ...
display(topic_table)

In [ ]:
# [작성] (4) 묶음 크기별 묶음 수를 세세요.

# 원본 묶음마다 행 수(묶음 크기)를 센 뒤, 묶음 크기별로 묶음이 몇 개인지 셉니다
bundle_sizes = ...
size_counts = ...
display(size_counts.rename_axis("묶음 크기").rename("묶음 수"))

In [ ]:
# [작성] (5) 주제 상한을 구하고, (6) 상한을 넘는 주제를 줄이세요.

# 상한: 범위 밖을 뺀 주제별 행 수 중앙값의 1.5배
TOPIC_CAP = ...
balanced = ...
print("주제 상한(행):", TOPIC_CAP, "| 균형 전:", len(decontaminated), "-> 균형 후:", len(balanced))

In [ ]:
# 주제별 행 수를 균형 전후로 나란히 봅니다
balanced_topic_counts = pd.DataFrame(balanced)["source"].value_counts().drop(index="범위 밖", errors="ignore")  # 범위 밖은 상한 대상이 아님
balance_table = pd.DataFrame({"균형 전": topic_counts, "균형 후": balanced_topic_counts}).rename_axis("주제").reset_index()
display(balance_table)

In [ ]:
# [자가채점] 구성 테이블, 주제 상한, 균형을 맞춘 행을 확인합니다.
expected_method_counts = {"문서 생성": 58, "패러프레이즈": 50, "역번역": 37, "기존 FAQ": 25, "범위 밖": 7}
expected_method_percent = {"문서 생성": 32.8, "패러프레이즈": 28.2, "역번역": 20.9, "기존 FAQ": 14.1, "범위 밖": 4.0}
expected_topic_counts = {"청약철회": 56, "환급·반환 비용": 46, "거래조건 표시·청약확인": 32, "통신판매중개": 20, "공급·배송": 16}
expected_paragraph_counts = {"거래조건 표시·청약확인": 7, "공급·배송": 4, "청약철회": 7, "통신판매중개": 7, "환급·반환 비용": 12}
expected_rows_per_paragraph = {"거래조건 표시·청약확인": 4.0, "공급·배송": 3.5, "청약철회": 6.1, "통신판매중개": 2.6, "환급·반환 비용": 3.5}
expected_size_counts = {1: 40, 2: 13, 3: 37}

# 방법별 테이블
assert isinstance(method_table, pd.DataFrame) and list(method_table.columns) == ["행 수", "비율(%)"], "method_table은 열이 \"행 수\", \"비율(%)\"인 데이터프레임입니다."
assert set(method_table.index) == set(expected_method_counts), f"method_table의 행 이름은 {list(expected_method_counts)}입니다. decontaminated의 method 열을 세세요."
for method in expected_method_counts:
    assert method_table.loc[method, "행 수"] == expected_method_counts[method], f"{method}의 행 수가 기대값 {expected_method_counts[method]}과 다릅니다. decontaminated의 method 열을 세세요."
    assert abs(method_table.loc[method, "비율(%)"] - expected_method_percent[method]) < 0.01, f"{method}의 비율이 기대값 {expected_method_percent[method]}과 다릅니다. 행 수를 len(decontaminated)로 나누고 100을 곱해 소수 첫째 자리로 반올림하세요."

# 주제별 테이블과 주제별 행 수
assert isinstance(topic_table, pd.DataFrame) and list(topic_table.columns) == ["행 수", "문단 수", "문단당 행 수"], "topic_table은 열이 \"행 수\", \"문단 수\", \"문단당 행 수\"인 데이터프레임입니다."
assert set(topic_table.index) == set(expected_topic_counts), f"topic_table의 행 이름은 주제 태그 {list(expected_topic_counts)}입니다. 범위 밖 행을 뺀 topic_groups로 만드세요."
assert list(topic_table["행 수"]) == sorted(topic_table["행 수"], reverse=True), "topic_table은 행 수가 많은 순서로 정렬합니다."
for topic in expected_topic_counts:
    assert topic_table.loc[topic, "행 수"] == expected_topic_counts[topic], f"{topic}의 행 수가 기대값 {expected_topic_counts[topic]}과 다릅니다. 주제별 묶음의 행 수를 세세요."
    assert topic_table.loc[topic, "문단 수"] == expected_paragraph_counts[topic], f"{topic}의 문단 수가 기대값 {expected_paragraph_counts[topic]}과 다릅니다. doc_id의 고유 수를 세세요."
    assert abs(topic_table.loc[topic, "문단당 행 수"] - expected_rows_per_paragraph[topic]) < 0.01, f"{topic}의 문단당 행 수가 기대값 {expected_rows_per_paragraph[topic]}과 다릅니다. doc_id가 있는 행 수를 문단 수로 나눠 소수 첫째 자리로 반올림하세요(기존 FAQ 행은 분자에 넣지 않습니다)."
assert isinstance(topic_counts, pd.Series) and "범위 밖" not in topic_counts.index, "topic_counts는 topic_table의 \"행 수\" 열입니다(범위 밖 없음)."
assert dict(topic_counts) == expected_topic_counts, f"주제별 행 수가 기대값 {expected_topic_counts}과 다릅니다. topic_table의 \"행 수\" 열을 꺼내세요."

# 묶음 크기별 묶음 수
assert dict(size_counts) == expected_size_counts, f"묶음 크기별 묶음 수가 기대값 {expected_size_counts}과 다릅니다. 범위 밖 행도 포함한 decontaminated_table의 origin_id별 행 수(bundle_sizes)를 한 번 더 세세요."
assert list(size_counts.index) == sorted(size_counts.index), "size_counts는 행 이름(묶음 크기)이 작은 순서로 정렬합니다."

# 주제 상한과 남은 행 수(미리 구한 값)
assert TOPIC_CAP == 48, f"TOPIC_CAP이 {TOPIC_CAP}입니다. topic_counts의 중앙값에 1.5를 곱해 round로 반올림하세요."
assert isinstance(balanced, list) and len(balanced) == 169, f"balanced가 {len(balanced)}행입니다. cap_by_topic에 decontaminated, TOPIC_CAP, SPLIT_SEED를 넣었는지 확인하세요."
position = 0
for row in decontaminated:
    if position < len(balanced) and row == balanced[position]:
        position += 1
assert position == len(balanced), "balanced는 decontaminated에서 행을 순서대로 고른 리스트입니다."
kept_ids = set()
for row in balanced:
    kept_ids.add(row["id"])
removed_ids = []
for row in decontaminated:
    if row["id"] not in kept_ids:
        removed_ids.append(row["id"])
assert sorted(removed_ids) == sorted(["c019-p", "c022-b", "c023-p", "c028-b", "c029-p", "c030-p", "c030-b", "c031-p"]), f"빠진 행이 {removed_ids}입니다. cap_by_topic의 시드로 SPLIT_SEED를 넣었는지 확인하세요."

# 균형의 성질: 범위 밖 행은 모두 남고, 주제마다 상한 이하
refusal_count = 0
balanced_totals = {}
for row in balanced:
    if row["source"] == "범위 밖":
        refusal_count += 1
    else:
        balanced_totals[row["source"]] = balanced_totals.get(row["source"], 0) + 1
assert refusal_count == expected_method_counts["범위 밖"], "범위 밖 행이 빠졌습니다. 범위 밖 행은 상한 대상이 아닙니다."
for topic in balanced_totals:
    assert balanced_totals[topic] <= TOPIC_CAP, f"{topic}이 {balanced_totals[topic]}행으로 상한 {TOPIC_CAP}행을 넘습니다."
print("✅ 통과!")

**결과 읽기**
- **방법별:** 증강 행 87행이 문서 생성 58행보다 많아 같은 답의 반복이 많고, 범위 밖 행은 7행(4.0%)으로 목표 3~5% 안입니다.
- **청약철회 56 -> 48행:** 상한 48행(중앙값 32의 1.5배)을 넘은 청약철회만 줄었습니다. 증강 행 8행만 빠지고 원본 묶음은 하나도 빠지지 않았습니다. 원본 Q&A를 같은 답의 반복보다 먼저 지키는 순서입니다.
- **적은 주제:** 공급·배송처럼 적은 주제를 같은 행 복제로 늘리면 같은 답의 반복이 늘어나므로, 생성 단계(과제 01의 질문 늘리기)에서 채웁니다.

## 5. chat 포맷으로 바꾸고 정합과 토큰 길이 확인하기

**배경**: 학습 텍스트가 추론 프롬프트로 정확히 시작해야 학습한 모델이 추론 때 학습에서 본 모양의 입력을 받고, 가장 긴 학습 텍스트의 토큰 수를 알아야 다음 단원에서 학습 설정의 최대 길이를 정할 수 있습니다.

**요구사항**
- (1) 4번에서 균형을 맞춘 `balanced`의 행마다 `dict(행)`으로 복사하고 `to_chat(행)["messages"]`를 `"messages"` 키로 더한 행을 **`chat_rows`** 에 담습니다. `to_chat`의 기본 system 문장은 `SYSTEM_PROMPT_COMMERCE`입니다.
- (2) `chat_tokenizer.apply_chat_template`로 `chat_rows` 첫 행의 세 메시지(셀의 `sample_messages`)를 모두 넣은 학습 텍스트(문자열) **`training_text`** 를 만듭니다.
- (3) 마지막 assistant 메시지를 빼고 모델이 답할 자리를 연 추론 프롬프트(문자열) **`inference_prompt`** 를 만듭니다.
- (4) `training_text`가 `inference_prompt`로 시작하는지를 **`aligned`**(bool)에 담습니다.
- (5) `training_text`에서 `inference_prompt` 뒤에 남는 부분을 **`answer_part`**(문자열)에 담습니다.
- (6) `chat_rows`의 행마다 (2)와 같은 방법으로 행의 `messages`에서 학습 텍스트를 만들고, 특수 토큰을 다시 더하지 않게 토큰화한 `input_ids`의 길이를 `chat_rows`와 같은 순서로 담은 정수 리스트 **`token_counts`** 를 만듭니다.
- (7) `token_counts` 가운데 가장 큰 값(정수)을 **`max_tokens`** 에 담습니다. 다음 단원에서 학습 설정의 최대 길이를 정할 때 쓰는 수입니다.

**예시**: 남는 부분은 첫 행의 답 문장 뒤에 끝 토큰과 줄바꿈이 붙은 `'… 넣어야 합니다.<|im_end|>\n'` 모양입니다.

**확인 기준**
- `chat_rows`: `balanced`와 같은 수이고, 행마다 원래 키 뒤에 `messages`(system, user, assistant)가 더해졌으며 `balanced`의 행은 그대로입니다. system 메시지는 `SYSTEM_PROMPT_COMMERCE`, user와 assistant 메시지는 그 행의 질문과 답입니다.
- `training_text`, `inference_prompt`: 둘 다 문자열이고, `inference_prompt`는 답할 자리와 빈 생각 블록인 `"<|im_start|>assistant\n<think>\n\n</think>\n\n"`으로 끝나며 답 문장이 없습니다.
- `aligned`, `answer_part`: `aligned`가 `True`, `answer_part`가 첫 행의 답 + `"<|im_end|>\n"`입니다.
- `token_counts`, `max_tokens`: `token_counts`가 `chat_rows`와 같은 수의 양의 정수 리스트이고, `max_tokens`가 그 가운데 가장 큰 값이며 자가채점 셀의 기대값과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 chat 포맷과 chat template 정합과 같은 순서입니다. 균형을 맞춘 행을 chat 행으로 바꾸고, 첫 행으로 두 문자열의 앞부분이 같은지 비교한 뒤, 같은 방법으로 모든 행의 토큰 수를 셉니다.

세부구현:
1. 학습 텍스트는 문자열로 받도록 토큰화를 끄고, add_generation_prompt는 기본값으로 둡니다.
2. 추론 프롬프트는 메시지 리스트에서 마지막 하나를 뺀 슬라이스를 넣고, 답할 자리를 여는 인자를 켭니다.
3. 남는 부분은 추론 프롬프트 길이 뒤부터 자른 학습 텍스트입니다.
4. 학습 텍스트에 특수 토큰이 이미 들어 있어, 토큰 수를 셀 때는 특수 토큰을 더하지 않습니다.
```

</details>

In [ ]:
# [작성] (1) 균형을 맞춘 행을 chat 행으로 바꾸세요.

# 원래 키를 그대로 두고 messages를 더한 행
chat_rows = []
for row in balanced:
    ...
print("chat 행 수:", len(chat_rows))

In [ ]:
# [작성] (2), (3) 첫 행의 학습 텍스트와 추론 프롬프트를 만들고, (4), (5) 정합과 남는 부분을 확인하세요.

# 첫 chat 행의 세 메시지
sample_messages = chat_rows[0]["messages"]
display(sample_messages)

# (2) 학습 텍스트(세 메시지 모두)와 (3) 추론 프롬프트(마지막 assistant를 빼고 답할 자리를 엶)
training_text = ...
inference_prompt = ...

# (4) 앞부분 일치와 (5) 남는 부분
aligned = ...
answer_part = ...
print("앞부분 일치:", aligned)
print("남는 부분:", repr(answer_part))

In [ ]:
# [작성] (6) 행마다 학습 텍스트의 토큰 수를 세고, (7) 가장 큰 값을 구하세요.

# (6) 행마다 학습 텍스트를 만들어 토큰 수 세기
token_counts = []
for row in chat_rows:
    ...

# (7) 가장 긴 학습 텍스트의 토큰 수
max_tokens = ...
print("chat 행 수:", len(chat_rows), "| 가장 긴 학습 텍스트(토큰):", max_tokens)

In [ ]:
# [자가채점] chat 행, 첫 행의 두 문자열과 정합, 학습 텍스트의 토큰 수를 확인합니다.
assert len(chat_rows) == len(balanced), "chat_rows는 balanced의 행마다 하나씩입니다. 4번의 balanced를 차례로 바꾸세요."
chat_ids = []
for chat_row in chat_rows:
    chat_ids.append(chat_row["id"])
balanced_ids = []
for row in balanced:
    balanced_ids.append(row["id"])
assert chat_ids == balanced_ids, "chat_rows의 행 순서가 balanced와 다릅니다. balanced를 차례로 바꿔 담으세요."
for row, chat_row in zip(balanced, chat_rows):
    assert "messages" not in row, "balanced의 행에 messages가 생겼습니다. dict(row)로 복사한 행에 messages를 더하고, 준비 셀부터 다시 실행하세요."
    assert list(chat_row) == list(row) + ["messages"], f"{chat_row['id']}: chat 행은 원래 키를 그대로 두고 끝에 messages를 더합니다."
    roles = []
    for message in chat_row["messages"]:
        roles.append(message["role"])
    assert roles == ["system", "user", "assistant"], "messages는 system, user, assistant 세 메시지입니다. to_chat(row)[\"messages\"]를 넣으세요."
    assert chat_row["messages"][0]["content"] == SYSTEM_PROMPT_COMMERCE, "system 메시지는 SYSTEM_PROMPT_COMMERCE입니다."
    assert chat_row["messages"][1]["content"] == row["question"] and chat_row["messages"][2]["content"] == row["answer"], "user, assistant 메시지는 그 행의 질문과 답입니다."

# 첫 행의 학습 텍스트와 추론 프롬프트
first_row = balanced[0]
assert isinstance(training_text, str) and isinstance(inference_prompt, str), "training_text, inference_prompt는 문자열입니다. 토큰화를 끄고(tokenize=False) 만드세요."
assert SYSTEM_PROMPT_COMMERCE in training_text, "학습 텍스트에 SYSTEM_PROMPT_COMMERCE가 없습니다. sample_messages를 그대로 넣으세요."
assert inference_prompt.endswith("<|im_start|>assistant\n<think>\n\n</think>\n\n"), "추론 프롬프트는 모델이 답할 자리(<|im_start|>assistant 와 줄바꿈)와 빈 생각 블록으로 끝나야 합니다. 추론 프롬프트를 만들 때 답할 자리를 여는 인자를 켰는지 확인하세요."
assert first_row["answer"] not in inference_prompt, "추론 프롬프트에 답이 들어 있습니다. 마지막 assistant 메시지를 빼고 만드세요."
assert first_row["answer"] in training_text, "학습 텍스트에 답이 없습니다. 세 메시지를 모두 넣으세요."
assert aligned is True, "aligned가 True가 아닙니다. 학습 텍스트에는 답할 자리를 여는 인자를 켜지 않았는지, 두 문자열을 같은 메시지로 만들었는지 확인하세요."
assert answer_part == first_row["answer"] + "<|im_end|>\n", f"남는 부분이 {repr(answer_part)}입니다. training_text에서 inference_prompt 길이 뒤부터 자르세요."

# 학습 텍스트의 토큰 수(가장 큰 값은 미리 구한 값)
expected_max_tokens = 122
assert isinstance(token_counts, list) and len(token_counts) == len(chat_rows), "token_counts는 chat_rows의 행마다 하나씩, 같은 순서로 담은 정수 리스트입니다."
for count in token_counts:
    assert isinstance(count, int) and count > 0, f"token_counts에 {count}가 있습니다. 행마다 input_ids의 길이(0보다 큰 정수)를 넣으세요."
assert max_tokens == max(token_counts), "max_tokens는 token_counts 가운데 가장 큰 값입니다."
assert max_tokens == expected_max_tokens, f"가장 긴 학습 텍스트가 {max_tokens}토큰입니다. 기대값은 {expected_max_tokens}토큰입니다. 학습 텍스트를 (2)의 training_text와 같은 방법으로 만들고, 토큰화할 때 특수 토큰을 다시 더하지 않았는지 확인하세요."
print("✅ 통과!")

**결과 읽기**
- **앞부분 일치 `True`:** 학습과 추론에 같은 토크나이저와 같은 system 문장을 썼다는 뜻입니다. system 문장이 다르면 `False`가 됩니다.
- **남는 부분:** 첫 행(`c001`)의 답과 끝 토큰입니다. 모델은 이 부분에서 답과 답을 끝내는 자리를 함께 배웁니다.
- **최대 122토큰:** 다음 단원에서 학습 설정의 최대 길이를 이보다 짧게 잡으면 긴 행은 끝의 답이 잘린 채 학습됩니다.

## 6. 행 단위 분할의 누수를 확인하고 원본 단위로 나누기

**배경**: 같은 원본에서 늘린 행이 학습과 검증에 갈리면, 모델이 검증 질문의 다른 표현과 답을 이미 학습한 상태가 되어 검증 결과가 처음 보는 질문의 성능을 보여 주지 못합니다.

**요구사항**
- (1) 5번의 `chat_rows`를 `list(chat_rows)`로 복사해 `random.Random(SPLIT_SEED).shuffle`로 섞고, 앞 `round(행 수 * VAL_RATIO)`행을 **`row_val`** 에 담습니다.
- (2) 섞은 리스트의 나머지 행을 **`row_train`** 에 담습니다.
- (3) `count_leaked(row_train, row_val)`을 **`row_leaked`** 에 담습니다.
- (4) `split_by_group`에 `key="origin_id"`, `val_ratio=VAL_RATIO`, `seed=SPLIT_SEED`를 주어 `chat_rows`를 나눈 두 리스트를 **`train_rows`**, **`val_rows`** 에 담습니다.
- (5) `count_leaked(train_rows, val_rows)`를 **`group_leaked`** 에 담습니다.

**예시**: `c013`(원본), `c013-p`(패러프레이즈), `c013-b`(역번역)는 `origin_id`가 모두 `c013`입니다. 행 단위로 섞으면 `c013-p`만 검증에 갈 수 있지만, 원본 단위로 나누면 세 행이 늘 같은 쪽에 갑니다.

**확인 기준**
- `chat_rows`: 섞은 뒤에도 행 순서가 `balanced`와 같습니다(복사한 리스트를 섞음).
- 행 단위 분할: `row_val`이 34행이고 `row_leaked`가 자가채점 셀의 기대값과 같습니다.
- 원본 단위 분할: `train_rows`, `val_rows`의 수가 자가채점 셀의 기대값과 같고 `group_leaked`가 0입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 행 단위 분할의 누수, 원본 단위 분할과 같은 순서입니다.

세부구현:
1. 섞기는 복사한 리스트에 합니다. random.Random(SPLIT_SEED)로 만든 난수 생성기의 shuffle은 리스트를 그 자리에서 섞고 None을 돌려줍니다.
2. 검증 행 수는 round로 반올림한 정수입니다. 슬라이스로 앞부분과 나머지를 나눕니다.
3. split_by_group은 (학습 행 리스트, 검증 행 리스트) 두 값을 돌려주므로 쉼표로 두 이름에 나눠 담습니다.
```

</details>

In [ ]:
# [작성] (1)~(3) 행 단위로 섞어 나누고 누수를 세세요.

# (1), (2) 행 단위로 섞어 앞 VAL_RATIO만큼을 검증으로
shuffled_rows = list(chat_rows)
...
row_val = ...
row_train = ...

# (3) 검증 행 가운데 같은 원본이 학습에도 있는 행 수
row_leaked = ...
print("행 단위 분할: 검증", len(row_val), "행 가운데 누수", row_leaked, "행")

In [ ]:
# [작성] (4), (5) 원본 Q&A(origin_id) 묶음 단위로 나누고 누수를 세세요.
train_rows, val_rows = ...
group_leaked = ...
print("원본 단위 분할: 학습", len(train_rows), "행 | 검증", len(val_rows), "행 | 누수", group_leaked, "행")

In [ ]:
# [자가채점] 행 단위 분할과 원본 단위 분할의 수와 누수를 확인합니다.
chat_ids = []
for chat_row in chat_rows:
    chat_ids.append(chat_row["id"])
balanced_ids = []
for row in balanced:
    balanced_ids.append(row["id"])
assert chat_ids == balanced_ids, "chat_rows의 행 순서가 balanced와 다릅니다. (1)에서 chat_rows를 그대로 섞지 말고 list(chat_rows)로 복사한 리스트를 섞은 뒤, 5번 (1) 셀부터 다시 실행하세요."

# 행 단위 분할(미리 구한 값)
expected_row_leaked = 24
assert len(row_val) == 34 and len(row_train) == len(chat_rows) - 34, f"행 단위 분할은 검증 {34}행(round(len(chat_rows) * VAL_RATIO)), 나머지 학습입니다."
assert row_leaked == expected_row_leaked, f"행 단위 분할의 누수가 {row_leaked}입니다. 복사한 리스트를 random.Random(SPLIT_SEED).shuffle로 섞은 뒤 앞 {34}행을 검증으로 나눴는지 확인하세요."

# 원본 단위 분할(미리 구한 값)
assert len(train_rows) == 135 and len(val_rows) == 34, f"원본 단위 분할이 학습 {len(train_rows)}행, 검증 {len(val_rows)}행입니다. split_by_group에 key=\"origin_id\", val_ratio=VAL_RATIO, seed=SPLIT_SEED를 주었는지 확인하세요."
assert group_leaked == 0, "원본 단위로 나누면 누수가 0이어야 합니다. key로 origin_id를 넘겼는지 확인하세요."
print("✅ 통과!")

**결과 읽기**
- **행 단위 누수 24행:** 검증 34행 가운데 24행은 같은 원본의 다른 표현이 학습에 있어, 검증 점수가 부풉니다.
- **원본 단위 분할:** 묶음 90개 가운데 18개를 검증으로 보내 누수가 0입니다. 묶음 수로 나누므로 검증 행 비율(0.201)은 0.2에서 조금 벗어날 수 있습니다.

## 7. 학습·검증 파일과 데이터 카드 저장하기

**배경**: 다음 단원의 SFT 학습이 바로 읽을 수 있게 최종 데이터를 파일로 저장하고, 넘겨받는 사람이 출처와 처리 과정을 알 수 있게 데이터 카드를 남깁니다.

**요구사항**
- (1) `save_jsonl`로 `train_rows`를 `train_path`(`output/commerce_train.jsonl`)에, `val_rows`를 `val_path`(`output/commerce_val.jsonl`)에 저장합니다.
- (2) 최종 학습·검증 행 가운데 `method`가 `"범위 밖"`인 행의 비율(%)을 소수 첫째 자리로 반올림해 **`refusal_percent`** 에 담습니다.
- (3) 마크다운 문자열 **`dataset_card`** 를 f-string으로 만듭니다. 절은 `## 목적`, `## 출처`, `## 처리 단계`, `## 최종 규모`, `## 한계` 다섯 개이고, 숫자는 손으로 적지 않고 앞 문항의 변수로 채웁니다.
    - **출처:** `data/과제/commerce_law_source.json`(셀의 `law_source`)의 `법령명`과 `시행일자`, 고정 파일 행 수. 시행일자는 `law_source["시행일자"]`의 여덟 자리 숫자 문자열을 날짜 모양으로 바꾸지 않고 그대로 넣습니다.
    - **처리 단계(이 과제의 순서대로):** 아래 다섯 가지를 적습니다.
        - 품질 필터: 단계별 남은 수
        - 오염 검사: 기준과 뺀 원본 묶음 수
        - 구성 균형: `주제별 상한 48행`, `177건 -> 169건`처럼 나오는 줄(f-string 모양은 확인 기준)과, 다음 줄에 셀의 `balanced_text`로 균형 뒤 주제별 행 수
        - chat 포맷: system 문장과 학습 텍스트 최대 토큰 수(`max_tokens`로 `최대 N토큰` 모양)
        - 분할: 분할 설정(`origin_id`, `VAL_RATIO`, `SPLIT_SEED`)과 누수
    - **단위:** 교안 02의 데이터 카드처럼 단계별 행 수와 최종 규모는 `N건`으로 적습니다(주제 상한과 주제별 행 수는 `N행`).
    - **최종 규모:** `학습 데이터 N건, 검증 데이터 M건` 모양의 줄, `범위 밖(거절) 행 비율: P%` 모양의 줄(`refusal_percent`로), 거절 설계(고정 거절 답 하나, 범위 밖 주제 네 가지: 해외 직구 관세, 연말정산 같은 세금, 개인 투자, 이직 준비) 한 줄
    - **한계:** `- `로 시작하는 줄 두 줄 이상. 문단 하나 단위로 만들어 다른 문단의 조건이 빠진 답이 있을 수 있다는 점과, 질문 유형이 생성용 LLM이 붙인 라벨이라는 점을 포함합니다.
- (4) `dataset_card`를 `card_path`(`output/commerce_dataset_card.md`)에 저장합니다.

**확인 기준**
- 두 파일: 다시 읽은 행 수가 `train_rows`, `val_rows`의 행 수와 같고 행마다 `messages`가 있습니다.
- `refusal_percent`: 자가채점 셀에 적어 둔 기대값과 같습니다.
- `dataset_card`: 다섯 절 제목, `학습 데이터 {len(train_rows)}건`, `검증 데이터 {len(val_rows)}건`, `주제별 상한 {TOPIC_CAP}행`, `{len(decontaminated)}건 -> {len(balanced)}건`, `balanced_text`, `최대 {max_tokens}토큰`, `범위 밖(거절) 행 비율: {refusal_percent}%`, 법령 시행일자가 들어 있고, 카드 파일이 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 데이터 카드와 같은 모양입니다. 절 제목과 글은 f-string 안에 그대로 쓰고, 숫자 자리에는 중괄호로 변수나 len(...)을 넣습니다.

세부구현:
1. 범위 밖 비율은 최종 두 리스트를 이은 데이터프레임의 method 열을 "범위 밖"과 비교한 결과(True, False)의 평균에 100을 곱한 값입니다.
2. f-string 안에서 딕셔너리 값을 꺼낼 때는 바깥 따옴표와 다른 따옴표로 키를 씁니다.
3. 문자열을 파일로 저장할 때는 경로의 write_text에 utf-8 인코딩을 줍니다.
```

</details>

In [ ]:
# [작성] (1) 최종 학습, 검증 데이터를 JSONL로 저장하세요.
train_path = output_dir / "commerce_train.jsonl"
val_path = output_dir / "commerce_val.jsonl"

# save_jsonl로 train_rows, val_rows를 두 경로에 저장
...

print("저장:", train_path.name, len(train_rows), "행 |", val_path.name, len(val_rows), "행")

In [ ]:
# [작성] (2) 범위 밖 비율을 구하고, (3) 앞 문항의 변수로 데이터 카드를 채우고, (4) 저장하세요.

# 법령명과 시행일자(법령이 개정되면 데이터를 다시 만들어야 하므로 카드에 적습니다)
law_source = json.loads((data_dir / "과제" / "commerce_law_source.json").read_text(encoding="utf-8"))

# 균형 뒤 주제별 행 수를 "주제 N행" 줄로 잇습니다
balanced_lines = []
for topic, count in balanced_topic_counts.items():
    balanced_lines.append(f"{topic} {count}행")
balanced_text = ", ".join(balanced_lines)

# (2) 최종 학습, 검증 행에서 범위 밖 행의 비율(%)
final_frame = pd.DataFrame(train_rows + val_rows)
refusal_percent = ...

# (3) 데이터 카드
dataset_card = f"""# 온라인 쇼핑 상담 Q&A 파인튜닝 데이터셋

## 목적
- ...

## 출처
- ...

## 처리 단계
1. ...

## 최종 규모
- ...

## 한계
- ...
"""

# (4) 데이터셋 파일과 같은 폴더에 저장합니다
card_path = output_dir / "commerce_dataset_card.md"
...
print(dataset_card)

In [ ]:
# [자가채점] 저장한 두 파일, 범위 밖 비율, 데이터 카드를 확인합니다.
assert train_path.exists() and val_path.exists(), "commerce_train.jsonl, commerce_val.jsonl 파일이 없습니다. save_jsonl로 두 파일을 저장하세요."
reloaded_train = load_jsonl(train_path)
reloaded_val = load_jsonl(val_path)
assert len(reloaded_train) == len(train_rows), f"commerce_train.jsonl이 {len(reloaded_train)}행입니다. train_rows를 저장하세요."
assert len(reloaded_val) == len(val_rows), f"commerce_val.jsonl이 {len(reloaded_val)}행입니다. val_rows를 저장하세요."
for row in reloaded_train + reloaded_val:
    assert "messages" in row, "저장한 행에 messages가 없습니다. 6번에서 나눈 학습, 검증 데이터(5번의 chat 행)를 저장하세요."

# 범위 밖 비율(미리 구한 값)
assert abs(refusal_percent - 4.1) < 0.01, f"refusal_percent가 {refusal_percent}입니다. 최종 학습, 검증 행 가운데 method가 \"범위 밖\"인 행의 비율(%)을 소수 첫째 자리로 반올림하세요."

# 데이터 카드: 다섯 절, 구성 균형, 최종 규모, 범위 밖 비율, 시행일자
assert isinstance(dataset_card, str), "dataset_card는 마크다운 문자열입니다."
for heading in ["## 목적", "## 출처", "## 처리 단계", "## 최종 규모", "## 한계"]:
    assert heading in dataset_card, f"데이터 카드에 '{heading}' 절이 없습니다."
assert "..." not in dataset_card, "데이터 카드에 ...가 남아 있습니다. 절마다 내용을 채우세요."
assert f"학습 데이터 {len(train_rows)}건" in dataset_card and f"검증 데이터 {len(val_rows)}건" in dataset_card, "최종 규모 절에 '학습 데이터 N건, 검증 데이터 M건'을 변수로 채우세요."
assert f"주제별 상한 {TOPIC_CAP}행" in dataset_card and f"{len(decontaminated)}건 -> {len(balanced)}건" in dataset_card, f"처리 단계 절의 구성 균형에 '주제별 상한 {TOPIC_CAP}행'과 '{len(decontaminated)}건 -> {len(balanced)}건'이 그대로 들어가게, f-string에 주제별 상한 {{TOPIC_CAP}}행과 {{len(decontaminated)}}건 -> {{len(balanced)}}건을 쓰세요."
assert balanced_text in dataset_card, "처리 단계 절의 구성 균형 다음 줄에 균형 뒤 주제별 행 수(balanced_text)를 넣으세요."
assert f"최대 {max_tokens}토큰" in dataset_card, "처리 단계 절에 학습 텍스트 최대 토큰 수를 '최대 N토큰' 모양으로 max_tokens 변수로 채우세요."
assert f"범위 밖(거절) 행 비율: {refusal_percent}%" in dataset_card, "최종 규모 절에 '범위 밖(거절) 행 비율: P%' 줄을 refusal_percent로 채우세요."
assert law_source["시행일자"] in dataset_card, "출처 절에 법령 시행일자(law_source의 시행일자)를 넣으세요."

# 한계 절: "- "로 시작하는 줄이 두 줄 이상
limit_lines = 0
in_limit = False
for line in dataset_card.splitlines():
    if line.startswith("## "):
        in_limit = False
        if line.strip() == "## 한계":
            in_limit = True
    elif in_limit and line.startswith("- "):
        limit_lines += 1
assert limit_lines >= 2, f"한계 절의 '- '로 시작하는 줄이 {limit_lines}줄입니다. 두 줄 이상 쓰세요."
assert card_path.exists() and card_path.read_text(encoding="utf-8") == dataset_card, "dataset_card를 output/commerce_dataset_card.md에 저장하세요."
print("✅ 통과!")

**결과 읽기**
- **남겨 둔 키:** 행마다 `messages`와 함께 `id`, `origin_id`, `source`, `method`가 남아, 학습 뒤 틀린 답을 원본까지 되짚을 수 있습니다.
- **범위 밖 4.1%:** 너무 많으면 답할 수 있는 질문까지 거절하는 모델이 되므로 몇 퍼센트로 둡니다.
- **카드의 숫자:** 변수로 채웠으므로 고정 파일이 바뀌어도 카드가 실제와 같습니다.